# HeritageAI — Google Colab Free GPU Worker for Stable Diffusion Inpainting

This notebook runs the official **Stable Diffusion Inpainting (v1.5)** model on a free Google Colab **T4 GPU** and exposes an HTTP endpoint for the HeritageAI web application.

### Instructions:
1. In the menu, click **Runtime > Change runtime type**, select **T4 GPU**, then click Save.
2. Run all cells below (Ctrl+F9).
3. Copy the generated public tunnel URL into `backend/.env` as `COLAB_SD_URL=https://...`

In [ ]:
# 1. Install dependencies
!pip install -q diffusers transformers accelerate torch torchvision uvicorn fastapi pyngrok nest_asyncio

In [ ]:
# 2. Load Stable Diffusion Inpainting Pipeline
import torch
from diffusers import StableDiffusionInpaintPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'})")

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
    requires_safety_checker=False
).to(device)

pipe.enable_attention_slicing()
print("Model successfully loaded and ready!")

In [ ]:
# 3. Run FastAPI Inpainting Server
import base64
import io
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio

app = FastAPI()

class InpaintRequest(BaseModel):
    image_base64: str
    mask_base64: str
    prompt: str
    steps: int = 25

@app.post("/inpaint")
def inpaint(req: InpaintRequest):
    img_bytes = base64.b64decode(req.image_base64)
    msk_bytes = base64.b64decode(req.mask_base64)
    
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    msk = Image.open(io.BytesIO(msk_bytes)).convert("L")
    orig_size = img.size
    
    img_r = img.resize((512, 512), Image.LANCZOS)
    msk_r = msk.resize((512, 512), Image.NEAREST)
    
    neg_prompt = "blurry, low quality, artifacts, watermark, text, modern, disfigured"
    result = pipe(
        prompt=req.prompt,
        negative_prompt=neg_prompt,
        image=img_r,
        mask_image=msk_r,
        num_inference_steps=req.steps,
        guidance_scale=7.5,
    ).images[0]
    
    res_orig = result.resize(orig_size, Image.LANCZOS)
    buf = io.BytesIO()
    res_orig.save(buf, format="PNG")
    
    return {"restored_image_base64": base64.b64encode(buf.getvalue()).decode()}

# Expose via localtunnel or pyngrok
print("Starting tunnel...")
!npm install -g localtunnel
import subprocess
import time
tunnel = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)
time.sleep(3)
for line in tunnel.stdout:
    print("YOUR COLAB_SD_URL:", line.decode().strip())
    break

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)